# 固定股票池量价因子风险平价回测

本示例使用平安银行和浦发银行组成固定股票池。策略只使用前一交易日及更早的数据，在当日开盘调仓，并在 2025-04-01 清仓。

## 策略规则

- 量价因子：`20 日价格动量 × 5 日成交量均值 / 20 日成交量均值`。
- 选股：因子大于 0 的股票进入组合。
- 风险平价：入选股票按 `1 / 20 日收益率标准差` 归一化配置权重。
- 调仓：按当日开盘价计算目标股数，向下取整到 100 股；先卖后买。
- 撮合：使用开盘价限价单。日频 `matchingMode=2` 按开盘价撮合，同时避免市价买单按涨停价冻结资金导致拒单。

In [1]:
import hashlib
import sys
from pathlib import Path

from dotenv import load_dotenv
from IPython.display import display

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
load_dotenv(project_root / ".env")
load_dotenv(project_root.parent / ".env")

from runtime import run_backtest
from runtime.utils import logger

logger.remove()
logger.add(sys.stderr, level="INFO")

2

## 数据集

`lookback=60D` 为 20 日动量和 20 个日收益率提供足够历史。数据不复权，与聚宽 `fq=None` 保持一致。

In [2]:
dataset_query = {
    "start_date": "2025-01-01",
    "end_date": "2025-04-02",
    "lookback": "60D",
    "codes": ["000001.SZ", "600000.SH"],
    "derivatives": {
        "return_1d": {"type": "TS", "op": "unary.pct_change", "fields": {"col": "close"}, "params": {"periods": 1}},
        "momentum_20d": {"type": "TS", "op": "unary.pct_change", "fields": {"col": "close"}, "params": {"periods": 20}},
        "volume_mean_5d": {"type": "TS", "op": "unary.rolling_mean", "fields": {"col": "vol"}, "params": {"window": 5, "min_periods": 5}},
        "volume_mean_20d": {"type": "TS", "op": "unary.rolling_mean", "fields": {"col": "vol"}, "params": {"window": 20, "min_periods": 20}},
        "volume_ratio": {"type": "DIRECT", "op": "binary.div", "fields": {"left": "volume_mean_5d", "right": "volume_mean_20d"}, "params": {}},
        "price_volume_factor": {"type": "DIRECT", "op": "binary.mul", "fields": {"left": "momentum_20d", "right": "volume_ratio"}, "params": {}},
        "volatility_20d": {"type": "TS", "op": "unary.rolling_std", "fields": {"col": "return_1d"}, "params": {"window": 20, "min_periods": 20}},
    },
    "filters": [],
}

print("股票池:", dataset_query["codes"])
print("量价因子: momentum_20d * volume_ratio")

股票池: ['000001.SZ', '600000.SH']
量价因子: momentum_20d * volume_ratio


## 回测回调

`backtest::getLastData` 只读取当前消息日期之前的最后一个截面。`dailyRecords` 保存每只股票每天的信号、权重、开盘价、权益和目标股数，供逐日核对。

In [3]:
callbacks = {
    "initialize": '''
        def initialize(mutable context) {
            context["dailyRecords"] = table(
                1000:0,
                `executionDate`signalDate`symbol`factor`volatility`weight`open`equity`currentQuantity`targetQuantity,
                [DATE, DATE, SYMBOL, DOUBLE, DOUBLE, DOUBLE, DOUBLE, DOUBLE, LONG, LONG]
            )
            print("[Strategy][INFO] initialize")
        }
    ''',
    "onBar": '''
        def onBar(mutable context, message, indicator) {
            executionDate = date(message.tradeTime[0])
            print("[Strategy][INFO] onBar date=" + string(executionDate) + " rows=" + string(message.rows()))
            if (executionDate > 2025.03.31) {
                for (index in 0..(message.rows() - 1)) {
                    position = Backtest::getPosition(context.engine, message.symbol[index], "stock")["longPosition"]
                    if (count(position) > 0 && position[0] > 0) {
                        print("[Strategy][INFO] submit side=SELL reason=liquidate symbol=" + string(message.symbol[index]) + " quantity=" + string(long(position[0])) + " price=" + string(message.open[index]))
                        Backtest::submitOrder(context.engine, (message.symbol[index], context.tradeTime, 5, message.open[index], long(position[0]), 3), "liquidate")
                    }
                }
                return
            }
            signal = backtest::getLastData(context, message, false)
            if (signal.rows() == 0) {
                print("[Strategy][INFO] skip date=" + string(executionDate) + " reason=no_previous_signal")
                return
            }
            rowCount = message.rows()
            risks = take(0.0, rowCount)
            factors = take(double(NULL), rowCount)
            volatilities = take(double(NULL), rowCount)
            signalCodes = string(signal.code)
            for (index in 0..(rowCount - 1)) {
                code = strReplace(strReplace(string(message.symbol[index]), ".XSHE", ".SZ"), ".XSHG", ".SH")
                signalIndex = find(signalCodes, code)
                if (signalIndex < signal.rows()) {
                    factors[index] = signal.price_volume_factor[signalIndex]
                    volatilities[index] = signal.volatility_20d[signalIndex]
                    if (!isNull(factors[index]) && factors[index] > 0 && !isNull(volatilities[index]) && volatilities[index] > 0) risks[index] = 1.0 / volatilities[index]
                }
            }
            riskSum = sum(risks)
            weights = iif(riskSum > 0, risks / riskSum, risks)
            currentQuantities = take(0l, rowCount)
            for (index in 0..(rowCount - 1)) {
                position = Backtest::getPosition(context.engine, message.symbol[index], "stock")["longPosition"]
                if (count(position) > 0) currentQuantities[index] = long(position[0])
            }
            equity = Backtest::getAvailableCash(context.engine, "stock") + sum(double(currentQuantities) * message.open)
            targetQuantities = take(0l, rowCount)
            for (index in 0..(rowCount - 1)) {
                if (weights[index] > 0) targetQuantities[index] = long(floor(equity * weights[index] / message.open[index] / 100.0)) * 100l
            }
            print("[Strategy][INFO] rebalance date=" + string(executionDate) + " signalDate=" + string(date(signal.time[0])) + " equity=" + string(equity) + " riskSum=" + string(riskSum))
            for (index in 0..(rowCount - 1)) {
                difference = targetQuantities[index] - currentQuantities[index]
                if (difference < 0) {
                    print("[Strategy][INFO] submit side=SELL reason=rebalance symbol=" + string(message.symbol[index]) + " quantity=" + string(-difference) + " price=" + string(message.open[index]))
                    Backtest::submitOrder(context.engine, (message.symbol[index], context.tradeTime, 5, message.open[index], -difference, 3), "riskParitySell")
                }
            }
            for (index in 0..(rowCount - 1)) {
                difference = targetQuantities[index] - currentQuantities[index]
                print("[Strategy][INFO] target symbol=" + string(message.symbol[index]) + " factor=" + string(factors[index]) + " volatility=" + string(volatilities[index]) + " weight=" + string(weights[index]) + " current=" + string(currentQuantities[index]) + " target=" + string(targetQuantities[index]))
                if (difference > 0) {
                    print("[Strategy][INFO] submit side=BUY reason=rebalance symbol=" + string(message.symbol[index]) + " quantity=" + string(difference) + " price=" + string(message.open[index]))
                    Backtest::submitOrder(context.engine, (message.symbol[index], context.tradeTime, 5, message.open[index], difference, 1), "riskParityBuy")
                }
                tableInsert(context["dailyRecords"], executionDate, date(signal.time[0]), message.symbol[index], factors[index], volatilities[index], weights[index], message.open[index], equity, currentQuantities[index], targetQuantities[index])
            }
        }
    ''',
    "onOrder": '''
        def onOrder(mutable context, event) {
            details = event[0]
            print("[Strategy][INFO] onOrder orderId=" + string(details["orderId"]) + " symbol=" + string(details["symbol"]) + " status=" + string(details["status"]) + " direction=" + string(details["direction"]) + " quantity=" + string(details["qty"]) + " traded=" + string(details["tradeQty"]) + " price=" + string(details["price"]) + " label=" + string(details["label"]))
        }
    ''',
    "onTrade": '''
        def onTrade(mutable context, event) {
            details = event[0]
            print("[Strategy][INFO] onTrade orderId=" + string(details["orderId"]) + " symbol=" + string(details["symbol"]) + " direction=" + string(details["direction"]) + " quantity=" + string(details["tradeQty"]) + " price=" + string(details["tradePrice"]) + " fee=" + string(details["totalFee"]))
        }
    ''',
    "finalize": '''
        def finalize(mutable context) {
            print("[Strategy][INFO] finalize dailyRecords=" + string(context["dailyRecords"].rows()))
        }
    ''',
}

print("回调:", list(callbacks))

回调: ['initialize', 'onBar', 'onOrder', 'onTrade', 'finalize']


## 执行并读取惰性结果

策略回调通过 `print` 输出 `[Strategy][INFO]` 日志；DolphinDB session 会把这些消息转发到当前 Notebook 的 INFO 输出。

In [4]:
engine_name = "volume_price_risk_parity"
with run_backtest(dataset_query, callbacks, name=engine_name, config={"cash": 100_000.0}) as backtest_result:
    context = backtest_result.context
    daily_records = context["dailyRecords"]
    trade_details = backtest_result.trade_details
    daily_portfolios = backtest_result.daily_portfolios
    return_summary = backtest_result.return_summary

filled_trades = trade_details[trade_details["orderStatus"] == 1].reset_index(drop=True)
print("逐股记录数:", len(daily_records))
print("成交数:", len(filled_trades))
display(daily_records.head(8))
display(filled_trades.tail(10))
display(daily_portfolios.tail(5))
display(return_summary)

2026-08-01 00:35:20.976 | INFO     | runtime.database.session:create_session:43 - DolphinDB: 1.13.198.44:8848
2026-08-01 00:35:21.126 | INFO     | runtime.apps.query.api:build_query_table:65 - session.run: 加载 query 模块
2026-08-01 00:35:21.159 | INFO     | runtime.database.session:has_session_variable:37 - session.run: 检查变量 coreBacktestSourceData 是否存在
2026-08-01 00:35:21.166 | INFO     | runtime.apps.query.api:build_query_table:69 - session.run: 查询基础因子表 coreBacktestSourceData
2026-08-01 00:35:21.188 | INFO     | runtime.apps.query.api:build_query_table:119 - session.run: 整理基础因子表 coreBacktestSourceData
2026-08-01 00:35:21.194 | INFO     | runtime.apps.query.api:build_query_table:127 - session.run: 计算 coreBacktestComputedData 并生成 coreBacktestFilteredData
2026-08-01 00:35:21.202 | INFO     | runtime.apps.query.api:build_query_table:140 - session.run: 投影 coreBacktestFilteredData 生成 coreBacktestData
2026-08-01 00:35:21.215 | INFO     | runtime.apps.backtest.api:run_backtest:143 - session.run:

逐股记录数: 114
成交数: 30


,executionDate,signalDate,symbol,factor,volatility,weight,open,equity,currentQuantity,targetQuantity
0,2025-01-02,2024-12-31,000001.XSHE,0.020518,0.010667,0.577096,11.73,100000.0,0,4900
1,2025-01-02,2024-12-31,600000.XSHG,0.080052,0.014556,0.422904,10.30,100000.0,0,4100
2,2025-01-03,2025-01-02,000001.XSHE,-0.002989,0.011934,0.000000,11.44,97841.0,4900,0
3,2025-01-03,2025-01-02,600000.XSHG,0.046747,0.015041,1.000000,10.12,97841.0,4100,9600
4,2025-01-06,2025-01-03,000001.XSHE,-0.006006,0.011968,0.000000,11.38,97937.0,0,0
5,2025-01-06,2025-01-03,600000.XSHG,0.049748,0.014868,1.000000,10.13,97937.0,9600,9600
6,2025-01-07,2025-01-06,000001.XSHE,-0.021542,0.011155,0.000000,11.42,97745.0,0,0
7,2025-01-07,2025-01-06,600000.XSHG,0.052837,0.014919,1.000000,10.11,97745.0,9600,9600


,orderId,symbol,direction,sendTime,orderPrice,orderQty,tradeTime,tradePrice,tradeQty,orderStatus,label
20,21,000001.XSHE,1,2025-02-25 15:00:00,11.56,3300,2025-02-25 15:00:00,11.56,3300,1,riskParityBuy
21,22,000001.XSHE,3,2025-02-28 15:00:00,11.58,3200,2025-02-28 15:00:00,11.58,3200,1,riskParitySell
22,23,600000.XSHG,1,2025-02-28 15:00:00,10.23,3600,2025-02-28 15:00:00,10.23,3600,1,riskParityBuy
23,24,600000.XSHG,3,2025-03-03 15:00:00,10.19,3600,2025-03-03 15:00:00,10.19,3600,1,riskParitySell
24,25,000001.XSHE,1,2025-03-03 15:00:00,11.52,3200,2025-03-03 15:00:00,11.52,3200,1,riskParityBuy
25,26,000001.XSHE,3,2025-03-17 15:00:00,11.63,3900,2025-03-17 15:00:00,11.63,3900,1,riskParitySell
26,27,600000.XSHG,1,2025-03-17 15:00:00,10.55,4200,2025-03-17 15:00:00,10.55,4200,1,riskParityBuy
27,28,000001.XSHE,3,2025-03-18 15:00:00,11.52,4400,2025-03-18 15:00:00,11.52,4400,1,riskParitySell
28,29,600000.XSHG,1,2025-03-18 15:00:00,10.63,4900,2025-03-18 15:00:00,10.63,4900,1,riskParityBuy
29,30,600000.XSHG,3,2025-04-01 15:00:00,10.43,9100,2025-04-01 15:00:00,10.43,9100,1,liquidate


,tradeDate,floatingPnl,realizedPnl,totalPnl,cash,totalMarketValue,totalEquity,netValue,totalReturn,ratio,pnl,totalFee,frozenFunds
54,2025-03-27,-392.0,-3570.0,-3962.0,33.0,96005.0,96038.0,0.96038,-0.03962,0.007638,728.0,0.0,0.0
55,2025-03-28,-1393.0,-3570.0,-4963.0,33.0,95004.0,95037.0,0.95037,-0.04963,-0.010423,-1001.0,0.0,0.0
56,2025-03-31,-1484.0,-3570.0,-5054.0,33.0,94913.0,94946.0,0.94946,-0.05054,-0.000958,-91.0,0.0,0.0
57,2025-04-01,0.0,-5054.0,-5054.0,94946.0,0.0,94946.0,0.94946,-0.05054,0.000000,0.0,0.0,0.0
58,2025-04-02,0.0,-5054.0,-5054.0,94946.0,0.0,94946.0,0.94946,-0.05054,0.000000,0.0,0.0,0.0


,totalReturn,annualReturn,annualVolatility,annualSkew,annualKur,sharpeRatio,maxDrawdown,drawdownRatio,beta,alpha,annualExcessReturn,benchmarkReturn,turnoverRate,dailyWinningRate
0,-0.05054,-0.197284,0.169222,-0.652625,3.860035,-1.402201,0.052823,-3.734825,0.0,0.0,0.0,0.0,0.074074,0.45614


## 与聚宽逐日校验

聚宽使用相同股票池、公式、前一日信号、开盘价、100 股取整和零费率。两端将逐股记录格式化到固定精度后计算 SHA-256；哈希相同表示 114 条记录全部一致。

In [5]:
record_lines = [
    f"{row.executionDate:%Y-%m-%d}|{row.signalDate:%Y-%m-%d}|{row.symbol}|{row.factor:.10f}|{row.volatility:.10f}|{row.weight:.10f}|{row.open:.4f}|{row.equity:.4f}|{row.currentQuantity}|{row.targetQuantity}"
    for row in daily_records.itertuples(index=False)
]
record_hash = hashlib.sha256("\n".join(record_lines).encode("utf-8")).hexdigest()
joinquant_hash = "a725409d4691d39fc530f56710dd0c54be5a73cb07750c63dcb98c24f1e37aa2"
final_equity = float(daily_portfolios.iloc[-1]["totalEquity"])

print("local hash:    ", record_hash)
print("JoinQuant hash:", joinquant_hash)
print("final equity:   ", final_equity)
print("total return:   ", f"{final_equity / 100_000 - 1:.3%}")

local hash:     a725409d4691d39fc530f56710dd0c54be5a73cb07750c63dcb98c24f1e37aa2
JoinQuant hash: a725409d4691d39fc530f56710dd0c54be5a73cb07750c63dcb98c24f1e37aa2
final equity:    94945.99999999999
total return:    -5.054%


In [6]:
assert len(daily_records) == 114
assert record_hash == joinquant_hash
assert round(final_equity, 2) == 94_946.0
assert not filled_trades.empty
assert backtest_result.closed

print("Backtest example checks passed.")

Backtest example checks passed.


## 差异定位结论

初版两端从 2025-02-05 开始分叉。逐日回溯发现 2025-01-27 的本地市价买单被拒：股票市价单按涨停价冻结资金，冻结额超过可用现金；聚宽按开盘价成交。将 DolphinDB 订单改为开盘价限价单后，两端 114 条信号和目标持仓记录哈希完全一致，清仓后资金均为 94946 元，累计收益均为 -5.054%（聚宽界面显示 -5.05%）。